# 🟩 AI Wordle Player
This notebook uses **Claude (claude-sonnet-4-20250514)** to play Wordle using information-theory-based reasoning.

### How it works
1. You choose (or randomly pick) a secret 5-letter word
2. Claude picks a guess using entropy/information theory
3. Feedback is computed automatically (🟩 correct, 🟨 present, ⬛ absent)
4. Claude uses the feedback to narrow down candidates and guess again
5. Repeat until solved (or 6 guesses used)

### Setup
You need an Anthropic API key. Set it as an environment variable or paste it in the config cell below.
```
pip install anthropic
```

In [ ]:
# Install dependency if needed
%pip install anthropic -q

In [ ]:
import os
import anthropic
import json
import random
import re

# ──────────────────────────────────────────────
# CONFIG — set your API key here OR as env var
# ──────────────────────────────────────────────
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")
MODEL   = "claude-sonnet-4-20250514"
MAX_GUESSES = 6

client = anthropic.Anthropic(api_key=API_KEY)
print("✅ Anthropic client ready.")

In [ ]:
# Common 5-letter words (representative sample for random mode)
WORD_LIST = [
    "CRANE","SLATE","TRACE","AROSE","IRATE","STARE","SNARE","LATER","ALTER","RAISE",
    "BREAD","CHAIN","DANCE","EARTH","FANCY","GRACE","HEART","INPUT","JUDGE","KNACK",
    "LEMON","MIRTH","NIGHT","OFTEN","PIANO","QUEEN","REACH","SHIRT","THINK","UNCLE",
    "VALID","WITCH","XENON","YIELD","ZONAL","ABBEY","BLINK","CLAMP","DRAPE","EMBER",
    "FLUTE","GROAN","HASTE","INDEX","JOUST","KNEEL","LOUPE","MAPLE","NOTCH","ONSET",
    "PERCH","QUOTA","ROBIN","SLUMP","TRAWL","ULTRA","VENOM","WHACK","EXPEL","YACHT",
    "ADOBE","BLAZE","CRISP","DROWN","EVOKE","FROTH","GLOOM","HINGE","IVORY","JOKER",
    "KNAVE","LOFTY","MAIZE","NERVE","ORBIT","PLUMB","QUIRK","REBEL","SCOFF","THYME",
    "USURP","VIVID","WRATH","OXIDE","YEARN","ABODE","BLUNT","CRAVE","DWARF","ELBOW",
    "FINCH","GLEAM","HIPPO","IGLOO","JUMBO","KARMA","LIBEL","MOOSE","NYMPH","OTTER",
    "PIXEL","QUALM","RIVET","SKIMP","TROUT","UPPER","VAPOR","WALTZ","PROXY","ZESTY",
    "MONEY","POWER","WATER","STONE","LIGHT","CLOUD","FLAME","SWORD","GHOST","PRIDE",
    "BRAVE","CLOWN","DELTA","EAGLE","FAITH","GLASS","HONEY","IMAGE","JEWEL","MAGIC",
    "NERVE","OCEAN","PLANT","QUIET","RIVER","SMILE","TOWER","URBAN","VOICE","WORLD"
]
print(f"✅ Word list loaded: {len(WORD_LIST)} words.")

In [ ]:
def get_feedback(guess: str, secret: str) -> list[int]:
    """Return feedback: 2=correct, 1=present, 0=absent."""
    fb = [0] * 5
    secret_chars = list(secret)
    used = [False] * 5

    # First pass: correct positions
    for i in range(5):
        if guess[i] == secret[i]:
            fb[i] = 2
            used[i] = True

    # Second pass: present but wrong position
    for i in range(5):
        if fb[i] == 2:
            continue
        for j in range(5):
            if not used[j] and guess[i] == secret_chars[j]:
                fb[i] = 1
                used[j] = True
                break
    return fb


def render_feedback(guess: str, fb: list[int]) -> str:
    """Render colored emoji string."""
    icons = {2: "🟩", 1: "🟨", 0: "⬛"}
    return " ".join(guess) + "  " + "".join(icons[f] for f in fb)


def render_board(history: list[dict]) -> str:
    """Print the current board state."""
    lines = ["\n" + "─" * 30]
    for i, entry in enumerate(history):
        lines.append(f"  Guess {i+1}: {render_feedback(entry['word'], entry['feedback'])}")
    lines.append("─" * 30)
    return "\n".join(lines)


def build_history_prompt(history: list[dict]) -> str:
    """Format guess history for the AI prompt."""
    rows = []
    for entry in history:
        fb_words = ["green" if f==2 else "yellow" if f==1 else "gray" for f in entry["feedback"]]
        rows.append(f"  {entry['word']} → [{', '.join(fb_words)}]")
    return "\n".join(rows)


print("✅ Helper functions defined.")

In [ ]:
SYSTEM_PROMPT = """You are an expert Wordle solver using information theory.
Respond ONLY with valid JSON — no markdown, no backticks, no extra text.
JSON format: {"word": "XXXXX", "reasoning": "brief explanation"}"""

STRONG_OPENERS = ["CRANE", "SLATE", "TRACE", "IRATE", "STARE", "AROSE"]


def ask_claude(history: list[dict], guesses_left: int) -> dict:
    """Ask Claude for the next guess. Returns {word, reasoning}."""

    if not history:
        prompt = (
            f"You are playing Wordle. Choose your opening guess.\n"
            f"Strategy: maximise entropy — cover the most common letters in varied positions.\n"
            f"Strong openers: {', '.join(STRONG_OPENERS)}\n"
            f"Reply ONLY with JSON: {{\"word\":\"XXXXX\",\"reasoning\":\"...\"}}"
        )
    else:
        history_str = build_history_prompt(history)
        prompt = (
            f"You are playing Wordle. {guesses_left} guess(es) remaining.\n\n"
            f"Guess history (green=correct position, yellow=wrong position, gray=not in word):\n"
            f"{history_str}\n\n"
            f"Using this feedback:\n"
            f"  - Lock in any green letters at their positions\n"
            f"  - Yellow letters must appear elsewhere in the word\n"
            f"  - Gray letters are eliminated\n"
            f"  - Eliminate duplicate guesses\n\n"
            f"Pick the guess that maximises information gain among remaining candidates.\n"
            f"Reply ONLY with JSON: {{\"word\":\"XXXXX\",\"reasoning\":\"...\"}}"
        )

    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()
    clean = re.sub(r"```json|```", "", raw).strip()
    parsed = json.loads(clean)

    word = re.sub(r"[^A-Z]", "", parsed["word"].upper())[:5]
    reasoning = parsed.get("reasoning", "")
    return {"word": word, "reasoning": reasoning}


print("✅ AI guess function defined.")

In [ ]:
def play_wordle(secret: str = None, verbose: bool = True) -> dict:
    """
    Run a full AI Wordle game.
    
    Args:
        secret: 5-letter word to guess. If None, picks randomly.
        verbose: Print step-by-step output.
    
    Returns:
        result dict with keys: secret, guesses, solved, num_guesses
    """
    if secret is None:
        secret = random.choice(WORD_LIST)
    secret = secret.upper().strip()
    assert len(secret) == 5 and secret.isalpha(), "Secret must be exactly 5 alphabetic characters."

    history = []
    solved = False

    if verbose:
        print(f"\n{'='*40}")
        print(f"  🟩 AI WORDLE — Secret word hidden")
        print(f"{'='*40}")

    for attempt in range(1, MAX_GUESSES + 1):
        guesses_left = MAX_GUESSES - attempt + 1

        if verbose:
            print(f"\n⏳ Guess {attempt}/{MAX_GUESSES} — asking Claude...")

        result = ask_claude(history, guesses_left)
        word     = result["word"]
        reasoning = result["reasoning"]

        if verbose:
            print(f"   🤖 Claude picks: {word}")
            print(f"   💭 Reasoning: {reasoning}")

        fb = get_feedback(word, secret)
        history.append({"word": word, "feedback": fb})

        if verbose:
            print(render_board(history))

        if all(f == 2 for f in fb):
            solved = True
            if verbose:
                print(f"\n🎉 Solved in {attempt} guess{'es' if attempt > 1 else ''}! The word was: {secret}")
            break

    if not solved and verbose:
        print(f"\n❌ Failed after {MAX_GUESSES} guesses. The word was: {secret}")

    return {
        "secret": secret,
        "guesses": history,
        "solved": solved,
        "num_guesses": len(history)
    }


print("✅ Game engine defined. Ready to play!")

---
## ▶️ Play a game
Run the cell below. Change `SECRET_WORD` to any 5-letter word, or set it to `None` for a random word.

In [ ]:
SECRET_WORD = "PERCH"  # ← Change me! Or set to None for random.

result = play_wordle(secret=SECRET_WORD, verbose=True)

---
## 📊 Batch evaluation (optional)
Run the AI against multiple words and measure performance. This uses API credits — adjust `NUM_GAMES` as needed.

In [ ]:
NUM_GAMES = 10  # ← Adjust: each game uses ~3-6 API calls

test_words = random.sample(WORD_LIST, min(NUM_GAMES, len(WORD_LIST)))
results = []

print(f"Running {NUM_GAMES} games...\n")
for i, word in enumerate(test_words):
    r = play_wordle(secret=word, verbose=False)
    status = "✅" if r["solved"] else "❌"
    print(f"  Game {i+1:2d}: {word}  {status}  {r['num_guesses']} guess(es)")
    results.append(r)

# Summary stats
solved    = [r for r in results if r["solved"]]
win_rate  = len(solved) / len(results) * 100
avg_guesses = sum(r["num_guesses"] for r in solved) / len(solved) if solved else 0
dist = {i: sum(1 for r in solved if r["num_guesses"] == i) for i in range(1, MAX_GUESSES+1)}

print(f"\n{'─'*35}")
print(f"  Games played : {len(results)}")
print(f"  Win rate     : {win_rate:.0f}%")
print(f"  Avg guesses  : {avg_guesses:.2f} (wins only)")
print(f"\n  Guess distribution:")
for k, v in dist.items():
    bar = "█" * v
    print(f"    {k}: {bar} ({v})")
print(f"{'─'*35}")